In [1]:
import pandas as pd
import sqlite3
import os
import warnings

warnings.filterwarnings("ignore")

# Create export folder
os.makedirs("../data/exports", exist_ok=True)
os.makedirs("../scripts", exist_ok=True)

# Columns needed for SQL analytics
sql_cols = [
    "loan_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "issue_d",
    "purpose", "addr_state", "dti", "delinq_2yrs",
    "fico_range_low", "fico_range_high", "open_acc",
    "pub_rec", "revol_bal", "revol_util",
    "total_acc", "mort_acc", "target"
]

# Load raw modeling dataset for readable SQL analysis
df_sql = pd.read_csv(
    "../data/processed/df_model_raw.csv",
    usecols=sql_cols,
    low_memory=False
)

print("SQL dataset loaded successfully.")
print("Shape:", df_sql.shape)
print("Default rate:", round(df_sql["target"].mean() * 100, 2), "%")
print(df_sql.head())

SQL dataset loaded successfully.
Shape: (1348099, 24)
Default rate: 19.98 %
   loan_amnt        term  int_rate  installment grade sub_grade emp_length  \
0     3600.0   36 months     13.99       123.03     C        C4  10+ years   
1    24700.0   36 months     11.99       820.28     C        C1  10+ years   
2    20000.0   60 months     10.78       432.66     B        B4  10+ years   
3    10400.0   60 months     22.45       289.91     F        F1    3 years   
4    11950.0   36 months     13.44       405.18     C        C3    4 years   

  home_ownership  annual_inc verification_status  ... delinq_2yrs  \
0       MORTGAGE     55000.0        Not Verified  ...         0.0   
1       MORTGAGE     65000.0        Not Verified  ...         1.0   
2       MORTGAGE     63000.0        Not Verified  ...         0.0   
3       MORTGAGE    104433.0     Source Verified  ...         1.0   
4           RENT     34000.0     Source Verified  ...         0.0   

  fico_range_low fico_range_high  open_a

In [2]:
# Clean term column: "36 months" -> 36
df_sql["term"] = (
    df_sql["term"]
    .astype(str)
    .str.strip()
    .str.replace(" months", "", regex=False)
    .astype(int)
)

# Ensure int_rate is numeric
df_sql["int_rate"] = pd.to_numeric(
    df_sql["int_rate"].astype(str).str.replace("%", "", regex=False).str.strip(),
    errors="coerce"
)

# Convert issue date
df_sql["issue_d"] = pd.to_datetime(df_sql["issue_d"], format="%b-%Y", errors="coerce")

# Create date features
df_sql["issue_year"] = df_sql["issue_d"].dt.year
df_sql["issue_month"] = df_sql["issue_d"].dt.month

# Create average FICO score
df_sql["fico_avg"] = (df_sql["fico_range_low"] + df_sql["fico_range_high"]) / 2

# Create SQLite connection
conn = sqlite3.connect(":memory:")

# Load dataframe into SQLite table
df_sql.to_sql("loans", conn, if_exists="replace", index=False)

print("SQLite table created successfully.")
print("Rows:", len(df_sql))
print("Columns:", df_sql.shape[1])

# Check table sample
sample = pd.read_sql_query("SELECT * FROM loans LIMIT 5", conn)
sample

SQLite table created successfully.
Rows: 1348099
Columns: 27


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,mort_acc,target,issue_year,issue_month,fico_avg
0,3600.0,36,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,...,7.0,0.0,2765.0,29.7,13.0,1.0,0,2015,12,677.0
1,24700.0,36,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,...,22.0,0.0,21470.0,19.2,38.0,4.0,0,2015,12,717.0
2,20000.0,60,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,...,6.0,0.0,7869.0,56.2,18.0,5.0,0,2015,12,697.0
3,10400.0,60,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,Source Verified,...,12.0,0.0,21929.0,64.5,35.0,6.0,0,2015,12,697.0
4,11950.0,36,13.44,405.18,C,C3,4 years,RENT,34000.0,Source Verified,...,5.0,0.0,8822.0,68.4,6.0,0.0,0,2015,12,692.0


In [3]:
def run_query(sql, export_name=None):
    result = pd.read_sql_query(sql, conn)
    
    if export_name:
        result.to_csv(f"../data/exports/{export_name}.csv", index=False)
        print(f"Exported: {export_name}.csv ({len(result)} rows)")
    
    return result

print("SQL helper function created successfully.")

SQL helper function created successfully.


In [4]:
q1 = """
SELECT
    grade,
    sub_grade,
    COUNT(*) AS total_loans,
    SUM(target) AS total_defaults,
    ROUND(AVG(target) * 100, 2) AS default_rate_pct,
    ROUND(AVG(int_rate), 2) AS avg_interest_rate,
    ROUND(AVG(loan_amnt), 0) AS avg_loan_amount,
    ROUND(AVG(annual_inc), 0) AS avg_annual_income,
    ROUND(AVG(dti), 2) AS avg_dti
FROM loans
GROUP BY grade, sub_grade
ORDER BY grade, sub_grade
"""

q1_result = run_query(q1, "q1_grade_default_rate")

q1_result

Exported: q1_grade_default_rate.csv (35 rows)


,grade,sub_grade,total_loans,total_defaults,default_rate_pct,avg_interest_rate,avg_loan_amount,avg_annual_income,avg_dti
0,A,A1,43682,1409,3.23,5.54,14022.0,99427.0,14.14
1,A,A2,37190,1737,4.67,6.52,13378.0,89083.0,15.21
2,A,A3,38010,2094,5.51,7.12,13607.0,88388.0,15.79
3,A,A4,52255,3588,6.87,7.51,14048.0,85957.0,15.99
4,A,A5,64056,5386,8.41,8.20,14132.0,84513.0,16.44
5,B,B1,71206,7431,10.44,8.91,13214.0,80883.0,16.68
6,B,B2,74080,8421,11.37,9.91,13310.0,78446.0,17.07
7,B,B3,81901,10637,12.99,10.75,13478.0,76604.0,17.39
8,B,B4,83276,12360,14.84,11.49,13276.0,74334.0,17.64
9,B,B5,82639,13812,16.71,12.01,12896.0,72913.0,17.98


### Q1 Insight: Default Rate by Grade and Sub-grade

The SQL analysis shows a clear increase in default risk as loan quality worsens from A1 to G5. A1 loans have the lowest default rate at around 3.23%, while G5 loans have the highest default rate at around 52.66%.

Average interest rate also increases consistently across lower grades, from around 5.54% for A1 to around 28.30% for G5. This confirms that loan grade and sub-grade are strong risk indicators and that the platform applies risk-based pricing.

In [5]:
q2 = """
SELECT
    purpose,
    COUNT(*) AS total_loans,
    SUM(target) AS total_defaults,
    ROUND(AVG(target) * 100, 2) AS default_rate_pct,
    ROUND(AVG(int_rate), 2) AS avg_interest_rate,
    ROUND(AVG(loan_amnt), 0) AS avg_loan_amount,
    ROUND(SUM(loan_amnt), 0) AS total_exposure
FROM loans
GROUP BY purpose
HAVING COUNT(*) > 1000
ORDER BY default_rate_pct DESC
"""

q2_result = run_query(q2, "q2_purpose_risk")

q2_result

Exported: q2_purpose_risk.csv (12 rows)


,purpose,total_loans,total_defaults,default_rate_pct,avg_interest_rate,avg_loan_amount,total_exposure
0,small_business,15577,4652,29.86,15.93,15631.0,2.434776e+08
1,moving,9526,2229,23.40,15.15,7854.0,7.481668e+07
2,house,7298,1599,21.91,15.41,15356.0,1.120675e+08
3,medical,15614,3411,21.85,14.01,8992.0,1.403950e+08
4,debt_consolidation,781442,165327,21.16,13.62,15225.0,1.189752e+10
5,other,78301,16508,21.08,14.58,9813.0,7.683342e+08
6,vacation,9084,1744,19.20,13.70,6190.0,5.622632e+07
7,major_purchase,29550,5498,18.61,12.74,11817.0,3.491919e+08
8,home_improvement,87721,15576,17.76,12.80,14133.0,1.239738e+09
9,credit_card,295625,50057,16.93,11.79,14805.0,4.376783e+09


### Q2 Insight: Risky Loan Purposes

The SQL analysis shows that small business loans have the highest default rate at around 29.86%, making them the riskiest loan purpose by default percentage. Moving, house, medical, and debt consolidation loans also show relatively high default rates.

However, debt consolidation has the highest total exposure because it has the largest number of loans and the highest total funded amount. This means debt consolidation is not the highest-risk purpose by rate, but it is highly important from a portfolio exposure perspective.

This distinction is important for credit risk analysis because a bank must monitor both default rate and total exposure while making lending decisions.

In [6]:
q3 = """
SELECT
    grade,
    COUNT(*) AS total_loans,
    ROUND(SUM(loan_amnt), 0) AS total_exposure,
    ROUND(AVG(target) * 100, 2) AS default_rate_pct,
    ROUND(SUM(loan_amnt) * AVG(target) * 0.6, 0) AS expected_loss,
    ROUND(
        SUM(loan_amnt) * AVG(target) * 0.6
        / SUM(SUM(loan_amnt) * AVG(target) * 0.6)
        OVER () * 100
    , 2) AS pct_of_total_loss
FROM loans
GROUP BY grade
ORDER BY grade
"""

q3_result = run_query(q3, "q3_portfolio_exposure")

q3_result

Exported: q3_portfolio_exposure.csv (7 rows)


,grade,total_loans,total_exposure,default_rate_pct,expected_loss,pct_of_total_loss
0,A,235193,3.266594e+09,6.04,118450903.0,4.87
1,B,393102,5.202049e+09,13.40,418128238.0,17.19
2,C,382323,5.421290e+09,22.44,730022215.0,30.02
3,D,201657,3.075050e+09,30.38,560525712.0,23.05
4,E,94192,1.654585e+09,38.43,381524981.0,15.69
5,F,32306,6.148668e+08,45.15,166554194.0,6.85
6,G,9326,1.903213e+08,49.67,56716816.0,2.33


### Q3 Insight: Portfolio Exposure and Expected Loss by Grade

This SQL query estimates expected loss by combining total exposure, default rate, and a 60% Loss Given Default assumption. Grade C contributes the highest expected loss at around $730M, accounting for approximately 30.02% of total expected loss.

Although Grade G has the highest default rate at around 49.67%, its portfolio exposure is much smaller, so it contributes only around 2.33% of total expected loss. This shows that the highest default rate does not always mean the highest business loss.

From a business perspective, Grade C and Grade D are the most important risk segments to monitor because they combine large portfolio exposure with relatively high default rates.

In [7]:
q4 = """
SELECT
    issue_year,
    COUNT(*) AS total_loans,
    SUM(target) AS total_defaults,
    ROUND(AVG(target) * 100, 2) AS default_rate_pct,
    ROUND(AVG(int_rate), 2) AS avg_interest_rate,
    ROUND(AVG(dti), 2) AS avg_dti,
    ROUND(AVG(fico_avg), 0) AS avg_fico
FROM loans
WHERE issue_year IS NOT NULL
GROUP BY issue_year
ORDER BY issue_year
"""

q4_result = run_query(q4, "q4_vintage_analysis")

q4_result

Exported: q4_vintage_analysis.csv (12 rows)


,issue_year,total_loans,total_defaults,default_rate_pct,avg_interest_rate,avg_dti,avg_fico
0,2007,603,158,26.20,11.83,10.71,692.0
1,2008,2393,496,20.73,12.06,13.20,699.0
2,2009,5281,723,13.69,12.44,12.47,717.0
3,2010,12537,1757,14.01,11.99,13.10,715.0
4,2011,21721,3297,15.18,12.22,13.85,717.0
5,2012,53367,8644,16.20,13.64,16.66,703.0
6,2013,134804,21024,15.60,14.53,17.22,697.0
7,2014,223103,41162,18.45,13.66,17.94,694.0
8,2015,375546,75804,20.19,12.39,18.94,696.0
9,2016,293105,68252,23.29,13.09,18.79,697.0


### Q4 Insight: Vintage Analysis by Issue Year

The SQL vintage analysis shows that default rates vary across issue years. Loans issued in 2007 have the highest default rate at around 26.20%, while default rates were relatively lower during 2009–2013.

Default rates increased again in 2015–2017, reaching around 23.29% in 2016 and 23.13% in 2017. The year 2015 had the highest loan volume, showing strong portfolio expansion during that period.

This indicates that default risk is affected not only by borrower-level characteristics but also by time-based portfolio conditions and broader lending trends.

In [8]:
q5 = """
WITH yearly AS (
    SELECT
        issue_year,
        COUNT(*) AS loans_issued,
        SUM(target) AS defaults_that_year,
        ROUND(SUM(loan_amnt), 0) AS volume_issued
    FROM loans
    WHERE issue_year IS NOT NULL
    GROUP BY issue_year
)
SELECT
    issue_year,
    loans_issued,
    defaults_that_year,
    volume_issued,
    SUM(loans_issued) OVER (ORDER BY issue_year) AS cumulative_loans,
    SUM(defaults_that_year) OVER (ORDER BY issue_year) AS cumulative_defaults,
    SUM(volume_issued) OVER (ORDER BY issue_year) AS cumulative_volume,
    ROUND(
        SUM(defaults_that_year) OVER (ORDER BY issue_year) * 100.0
        / SUM(loans_issued) OVER (ORDER BY issue_year)
    , 2) AS cumulative_default_rate_pct
FROM yearly
ORDER BY issue_year
"""

q5_result = run_query(q5, "q5_cumulative_defaults")

q5_result

Exported: q5_cumulative_defaults.csv (12 rows)


,issue_year,loans_issued,defaults_that_year,volume_issued,cumulative_loans,cumulative_defaults,cumulative_volume,cumulative_default_rate_pct
0,2007,603,158,4.977475e+06,603,158,4.977475e+06,26.20
1,2008,2393,496,2.111925e+07,2996,654,2.609672e+07,21.83
2,2009,5281,723,5.192825e+07,8277,1377,7.802498e+07,16.64
3,2010,12537,1757,1.319926e+08,20814,3134,2.100175e+08,15.06
4,2011,21721,3297,2.616838e+08,42535,6431,4.717014e+08,15.12
5,2012,53367,8644,7.184110e+08,95902,15075,1.190112e+09,15.72
6,2013,134804,21024,1.982613e+09,230706,36099,3.172725e+09,15.65
7,2014,223103,41162,3.253490e+09,453809,77261,6.426215e+09,17.03
8,2015,375546,75804,5.498601e+09,829355,153065,1.192482e+10,18.46
9,2016,293105,68252,4.240362e+09,1122460,221317,1.616518e+10,19.72


### Q5 Insight: Cumulative Default Trend

This SQL query uses window functions to calculate cumulative loans, cumulative defaults, cumulative funded volume, and cumulative default rate over time.

The cumulative default rate started high at around 26.20% in 2007, reduced during the early years, and gradually increased again after 2014 as the portfolio expanded. By 2018, the cumulative default rate reached around 19.98%, matching the overall portfolio default rate.

This query is useful for tracking how the credit portfolio evolved over time and how default risk accumulated across yearly loan cohorts.

In [9]:
q6 = """
WITH ranked AS (
    SELECT
        grade,
        int_rate,
        dti,
        fico_avg,
        loan_amnt,
        target,
        NTILE(4) OVER (ORDER BY int_rate DESC) AS risk_quartile
    FROM loans
    WHERE int_rate IS NOT NULL
)
SELECT
    risk_quartile,
    CASE risk_quartile
        WHEN 1 THEN 'Q1 - Highest Risk'
        WHEN 2 THEN 'Q2 - High Risk'
        WHEN 3 THEN 'Q3 - Medium Risk'
        WHEN 4 THEN 'Q4 - Low Risk'
    END AS risk_label,
    COUNT(*) AS total_loans,
    ROUND(AVG(int_rate), 2) AS avg_int_rate,
    ROUND(AVG(dti), 2) AS avg_dti,
    ROUND(AVG(fico_avg), 0) AS avg_fico,
    ROUND(AVG(target) * 100, 2) AS default_rate_pct,
    ROUND(SUM(loan_amnt), 0) AS total_exposure
FROM ranked
GROUP BY risk_quartile
ORDER BY risk_quartile
"""

q6_result = run_query(q6, "q6_risk_quartiles")

q6_result

Exported: q6_risk_quartiles.csv (4 rows)


,risk_quartile,risk_label,total_loans,avg_int_rate,avg_dti,avg_fico,default_rate_pct,total_exposure
0,1,Q1 - Highest Risk,337025,19.70,20.44,685.0,34.30,5.567188e+09
1,2,Q2 - High Risk,337025,14.24,18.81,690.0,22.45,4.763047e+09
2,3,Q3 - Medium Risk,337025,11.37,17.80,697.0,15.58,4.464558e+09
3,4,Q4 - Low Risk,337024,7.66,16.05,721.0,7.59,4.629964e+09


### Q6 Insight: Borrower Risk Quartile Segmentation

This SQL query uses the `NTILE(4)` window function to divide borrowers into four risk quartiles based on interest rate. Q1 represents the highest-risk borrowers, while Q4 represents the lowest-risk borrowers.

The highest-risk quartile has a default rate of around 34.30%, while the lowest-risk quartile has a default rate of only around 7.59%. This shows that interest rate is a strong proxy for borrower risk.

The risk pattern is also supported by borrower characteristics. Q1 borrowers have higher average DTI, lower average FICO scores, and higher default rates, while Q4 borrowers have lower DTI, higher FICO scores, and lower default rates.

In [10]:
q7 = """
SELECT
    addr_state,
    COUNT(*) AS total_loans,
    ROUND(SUM(loan_amnt), 0) AS total_exposure,
    SUM(target) AS total_defaults,
    ROUND(AVG(target) * 100, 2) AS default_rate_pct,
    ROUND(AVG(dti), 2) AS avg_dti,
    ROUND(AVG(fico_avg), 0) AS avg_fico,
    RANK() OVER (
        ORDER BY AVG(target) DESC
    ) AS risk_rank
FROM loans
WHERE addr_state IS NOT NULL
GROUP BY addr_state
HAVING COUNT(*) > 500
ORDER BY default_rate_pct DESC
LIMIT 20
"""

q7_result = run_query(q7, "q7_state_exposure")

q7_result

Exported: q7_state_exposure.csv (20 rows)


,addr_state,total_loans,total_exposure,total_defaults,default_rate_pct,avg_dti,avg_fico,risk_rank
0,MS,6595,9.218970e+07,1722,26.11,20.53,696.0,1
1,NE,3592,4.793662e+07,906,25.22,20.14,697.0,2
2,AR,10062,1.357667e+08,2426,24.11,20.16,699.0,3
3,AL,16645,2.321572e+08,3934,23.63,20.13,698.0,4
4,OK,12298,1.752447e+08,2886,23.47,19.88,699.0,5
5,LA,15524,2.211876e+08,3598,23.18,19.34,700.0,6
6,NY,110097,1.573630e+09,24277,22.05,16.55,699.0,7
7,NV,20296,2.793486e+08,4459,21.97,18.52,696.0,8
8,FL,95843,1.315089e+09,20608,21.50,18.48,697.0,9
9,IN,21726,3.047970e+08,4656,21.43,19.75,697.0,10


### Q7 Insight: State-level Portfolio Exposure

This SQL query ranks states by default risk using the `RANK()` window function. Mississippi, Nebraska, Arkansas, Alabama, and Oklahoma show the highest default rates among states with sufficient loan volume.

However, states such as New York and Florida have much larger total exposure even though their default rates are slightly lower than the top-ranked states. This shows that geographic risk should be evaluated using both default rate and total exposure.

This analysis can support geographic risk monitoring and can later be reused for dashboard-level state risk visualization.

In [11]:
q8 = """
WITH monthly AS (
    SELECT
        issue_year,
        issue_month,
        COUNT(*) AS total_loans,
        SUM(target) AS total_defaults,
        ROUND(AVG(target) * 100, 2) AS default_rate_pct
    FROM loans
    WHERE issue_year IS NOT NULL
      AND issue_month IS NOT NULL
    GROUP BY issue_year, issue_month
)
SELECT
    issue_year,
    issue_month,
    total_loans,
    total_defaults,
    default_rate_pct,
    LAG(default_rate_pct) OVER (
        ORDER BY issue_year, issue_month
    ) AS prev_month_default_rate,
    ROUND(
        default_rate_pct -
        LAG(default_rate_pct) OVER (
            ORDER BY issue_year, issue_month
        )
    , 2) AS mom_change_pct
FROM monthly
ORDER BY issue_year, issue_month
"""

q8_result = run_query(q8, "q8_monthly_trend")

q8_result.tail(24)

Exported: q8_monthly_trend.csv (139 rows)


,issue_year,issue_month,total_loans,total_defaults,default_rate_pct,prev_month_default_rate,mom_change_pct
115,2017,1,16395,3720,22.69,24.94,-2.25
116,2017,2,13468,3127,23.22,22.69,0.53
117,2017,3,17011,3904,22.95,23.22,-0.27
118,2017,4,13104,3119,23.80,22.95,0.85
119,2017,5,16153,3868,23.95,23.80,0.15
120,2017,6,15235,3604,23.66,23.95,-0.29
121,2017,7,14959,3601,24.07,23.66,0.41
122,2017,8,15489,3547,22.90,24.07,-1.17
123,2017,9,13406,3350,24.99,22.90,2.09
124,2017,10,11926,2649,22.21,24.99,-2.78


### Q8 Insight: Month-over-Month Default Trend

This SQL query uses the `LAG()` window function to compare each month's default rate with the previous month. This helps identify whether portfolio default risk is increasing or decreasing month over month.

The 2017 monthly default rates remain mostly in the 21%–25% range, showing a relatively high-risk period. In 2018, the default rate declines sharply toward the later months.

However, the sharp decline in late 2018 should be interpreted carefully because recent loans may not have had enough time to fully mature into default outcomes. This highlights the importance of considering loan seasoning while analyzing credit risk trends.

In [15]:
q9 = """
WITH high_risk AS (
    SELECT *
    FROM loans
    WHERE dti > 25
      AND int_rate > 16
      AND fico_avg < 700
),
risk_summary AS (
    SELECT
        grade,
        purpose,
        COUNT(*) AS borrower_count,
        ROUND(AVG(target) * 100, 2) AS default_rate_pct,
        ROUND(AVG(loan_amnt), 0) AS avg_loan_amnt,
        ROUND(AVG(dti), 2) AS avg_dti,
        ROUND(AVG(int_rate), 2) AS avg_int_rate,
        ROUND(AVG(fico_avg), 0) AS avg_fico,
        ROUND(SUM(loan_amnt) * AVG(target) * 0.6, 0) AS expected_loss
    FROM high_risk
    GROUP BY grade, purpose
    HAVING COUNT(*) > 100
)
SELECT *
FROM risk_summary
ORDER BY default_rate_pct DESC
LIMIT 15
"""

q9_result = run_query(q9, "q9_high_risk_profile")

q9_result

Exported: q9_high_risk_profile.csv (15 rows)


,grade,purpose,borrower_count,default_rate_pct,avg_loan_amnt,avg_dti,avg_int_rate,avg_fico,expected_loss
0,G,other,213,57.28,14486.0,33.86,27.62,676.0,1060386.0
1,G,debt_consolidation,1899,56.77,20065.0,32.53,28.21,675.0,12977919.0
2,F,debt_consolidation,6332,53.55,19019.0,31.59,25.21,676.0,38695854.0
3,F,home_improvement,332,52.71,18318.0,30.93,25.19,676.0,1923429.0
4,G,credit_card,252,52.38,20102.0,31.82,28.36,675.0,1592069.0
5,F,credit_card,977,52.30,19340.0,32.00,25.42,676.0,5929791.0
6,E,house,138,49.28,12770.0,30.77,21.35,675.0,520998.0
7,F,major_purchase,109,48.62,16608.0,29.89,25.50,676.0,528143.0
8,E,small_business,231,47.19,13706.0,31.14,21.15,677.0,896376.0
9,F,other,614,46.58,13098.0,30.72,25.13,677.0,2247674.0


In [14]:
pd.read_sql_query("""
SELECT
    MIN(fico_avg) AS min_fico,
    MAX(fico_avg) AS max_fico,
    AVG(fico_avg) AS avg_fico
FROM loans
""", conn)

,min_fico,max_fico,avg_fico
0,612.0,847.5,698.162302


### Q9 Insight: High-risk Borrower Profile

This SQL query uses CTEs to define and profile a high-risk borrower segment. The high-risk segment is defined using three risk conditions: DTI greater than 25, interest rate greater than 16%, and FICO score below 700.

The results show that Grade G and Grade F borrowers have the highest default rates within this high-risk segment. For example, Grade G borrowers with loan purpose `other` have a default rate of around 57.28%, while Grade G debt consolidation borrowers have a default rate of around 56.77%.

Debt consolidation appears repeatedly among high-risk grades, especially in Grade E, F, and G. Although some segments have higher default rates, Grade E debt consolidation has a very large borrower count and expected loss, making it one of the most important segments from a business-risk perspective.

This analysis shows how SQL can be used to define business risk rules and identify borrower segments that require closer monitoring or stricter lending criteria.

In [16]:
q10 = """
WITH dti_buckets AS (
    SELECT *,
        CASE
            WHEN dti < 10 THEN '1. 0-10%'
            WHEN dti < 20 THEN '2. 10-20%'
            WHEN dti < 30 THEN '3. 20-30%'
            WHEN dti < 40 THEN '4. 30-40%'
            ELSE '5. 40%+'
        END AS dti_bucket
    FROM loans
    WHERE dti IS NOT NULL
)
SELECT
    dti_bucket,
    COUNT(*) AS total_loans,
    ROUND(AVG(target) * 100, 2) AS default_rate_pct,
    ROUND(AVG(int_rate), 2) AS avg_int_rate,
    ROUND(AVG(fico_avg), 0) AS avg_fico,
    ROUND(SUM(loan_amnt), 0) AS total_exposure,
    LAG(ROUND(AVG(target) * 100, 2)) OVER (
        ORDER BY dti_bucket
    ) AS prev_bucket_default_rate,
    ROUND(
        ROUND(AVG(target) * 100, 2) -
        LAG(ROUND(AVG(target) * 100, 2)) OVER (
            ORDER BY dti_bucket
        )
    , 2) AS rate_increase_vs_prev_bucket
FROM dti_buckets
GROUP BY dti_bucket
ORDER BY dti_bucket
"""

q10_result = run_query(q10, "q10_dti_bucket_analysis")

q10_result

Exported: q10_dti_bucket_analysis.csv (5 rows)


,dti_bucket,total_loans,default_rate_pct,avg_int_rate,avg_fico,total_exposure,prev_bucket_default_rate,rate_increase_vs_prev_bucket
0,1. 0-10%,246554,14.93,12.26,703.0,3.336119e+09,NaN,NaN
1,2. 10-20%,563149,17.86,12.82,698.0,8.227194e+09,14.93,2.93
2,3. 20-30%,409337,23.05,13.75,696.0,6.018110e+09,17.86,5.19
3,4. 30-40%,121915,29.09,15.28,695.0,1.719067e+09,23.05,6.04
4,5. 40%+,6770,30.55,16.69,699.0,1.178080e+08,29.09,1.46


### Q10 Insight: DTI Bucket Analysis

This SQL query groups borrowers into DTI buckets and uses the `LAG()` window function to compare each bucket's default rate with the previous bucket.

The results show a clear increase in default risk as DTI increases. Borrowers with DTI below 10% have a default rate of around 14.93%, while borrowers with DTI between 30% and 40% have a default rate of around 29.09%.

The largest increase happens between the 20–30% and 30–40% DTI buckets, where default risk increases by around 6.04 percentage points. This suggests that DTI above 30% can be considered an important warning signal in credit risk analysis.

This finding can support future credit policy decisions and risk scorecard rules.

In [17]:
all_queries = {
    "Q1_grade_default_rate": q1,
    "Q2_purpose_risk": q2,
    "Q3_portfolio_exposure": q3,
    "Q4_vintage_analysis": q4,
    "Q5_cumulative_defaults": q5,
    "Q6_risk_quartiles": q6,
    "Q7_state_exposure": q7,
    "Q8_monthly_trend": q8,
    "Q9_high_risk_profile": q9,
    "Q10_dti_bucket_analysis": q10
}

with open("../scripts/sql_analytics.sql", "w", encoding="utf-8") as f:
    for name, query in all_queries.items():
        f.write("-- " + "=" * 60 + "\n")
        f.write(f"-- {name}\n")
        f.write("-- " + "=" * 60 + "\n")
        f.write(query.strip())
        f.write("\n\n")

print("Saved: scripts/sql_analytics.sql")

Saved: scripts/sql_analytics.sql
